# Common setup

Run these cells first.

In [1]:
from pathlib import Path
import csv
import json
import shutil
import subprocess
import sys
from collections import Counter

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "scripts":
    PROJECT_ROOT = PROJECT_ROOT.parent
if PROJECT_ROOT.name == "vision" and PROJECT_ROOT.parent.name == "scripts":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

MANIFEST_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/manifests"
RAW_VIDEO_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/raw_videos"
CLIP_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/clips_5s"
MODEL_DIR = PROJECT_ROOT / "storage/vision/models"
SAMPLE_MANIFEST = MANIFEST_DIR / "sample_700_coarse_manifest.csv"
DOWNLOAD_MANIFEST = MANIFEST_DIR / "train_700_download_manifest.csv"
CLIP_MANIFEST = MANIFEST_DIR / "train_700_clip_manifest_5s.csv"

PER_LABEL = 700
SEED = 42
DEVICE = "auto"
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /workspace/SKN27-FINAL-3Team


In [3]:
def run_command(command, *, timeout=None):
    print("$", " ".join(map(str, command)))
    completed = subprocess.run(list(map(str, command)), cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=timeout)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
    return completed


## Install and environment check

In [4]:
run_command([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], timeout=3600)
usage = shutil.disk_usage(PROJECT_ROOT)
print("free_gb:", round(usage.free / 1024**3, 2))
run_command([sys.executable, "-c", "import torch; print('cuda_available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"])


$ /usr/bin/python -m pip install -r requirements.txt


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip

free_gb: 117889.3
$ /usr/bin/python -c import torch; print('cuda_available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
cuda_available: True
device: NVIDIA RTX A5000



CompletedProcess(args=['/usr/bin/python', '-c', "import torch; print('cuda_available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"], returncode=0, stdout='cuda_available: True\ndevice: NVIDIA RTX A5000\n', stderr='')

## Build or check downloaded-video manifest

In [5]:
if not SAMPLE_MANIFEST.exists():
    raise FileNotFoundError(f"sample manifest not found: {SAMPLE_MANIFEST}")

if not DOWNLOAD_MANIFEST.exists():
    run_command([
        sys.executable,
        "etl/vision/download_sampled_media.py",
        "--input", SAMPLE_MANIFEST,
        "--output", DOWNLOAD_MANIFEST,
        "--download-dir", RAW_VIDEO_DIR,
        "--label-column", "coarse_label",
        "--per-label", str(PER_LABEL),
        "--split", "",
    ], timeout=None)

with DOWNLOAD_MANIFEST.open("r", encoding="utf-8", newline="") as f:
    download_rows = list(csv.DictReader(f))
print("download_rows:", len(download_rows))
print("coarse_label_counts:", dict(Counter(row.get("coarse_label") for row in download_rows)))
print("download_status_counts:", dict(Counter(row.get("download_status") for row in download_rows)))


download_rows: 2800
coarse_label_counts: {'차대보행자': 700, '차대이륜차': 700, '차대자전거': 700, '차대차': 700}
download_status_counts: {'exists': 921, 'downloaded': 1879}


# Model 3: Service pipeline review

Run the practical service output path: YOLO evidence, 5-second clip policy, VideoMAE hint, final merge, supervisor handoff.

In [6]:
commands = [
    [sys.executable, "scripts/vision/check_raw_media.py"],
    [sys.executable, "ai/vision/pipeline.py"],
    [sys.executable, "ai/vision/models.py"],
    [sys.executable, "ai/vision/schemas.py"],
    [sys.executable, "ai/vision/visualize.py"],
    [sys.executable, "etl/vision/build_clip_candidates.py", "--short-video-sec", "5"],
    [sys.executable, "etl/vision/extract_video_clips.py", "--overwrite"],
    [sys.executable, "etl/vision/extract_videomae_frames.py", "--overwrite"],
    [sys.executable, "ai/vision/videomae_infer.py"],
    [sys.executable, "ai/vision/merge_analysis.py"],
    [sys.executable, "ai/vision/build_supervisor_handoff.py"],
]
for command in commands:
    run_command(command, timeout=None)


$ /usr/bin/python scripts/vision/check_raw_media.py
raw_dir: storage/vision/raw
found_media: 11
OK   storage/vision/raw/864.jpg | image 28x28 RGB
OK   storage/vision/raw/865.jpg | image 28x28 RGB
OK   storage/vision/raw/866.jpg | image 28x28 RGB
OK   storage/vision/raw/867.jpg | image 28x28 RGB
OK   storage/vision/raw/868.jpg | image 28x28 RGB
OK   storage/vision/raw/869.jpg | image 28x28 RGB
OK   storage/vision/raw/870.jpg | image 28x28 RGB
OK   storage/vision/raw/871.jpg | image 28x28 RGB
OK   storage/vision/raw/872.jpg | image 28x28 RGB
OK   storage/vision/raw/873.jpg | image 28x28 RGB
OK   storage/vision/raw/bb_3_190909_pedestrian_226_21450.mp4 | video 1920x1080 fps=15.00 frames=150
failed_media: 0

$ /usr/bin/python ai/vision/pipeline.py
source_video: storage/vision/raw/bb_3_190909_pedestrian_226_21450.mp4
total_frames: 150
fps: 15.0
keyframe_count: 5
output_path: storage/vision/outputs/keyframes_bb_3_190909_pedestrian_226_21450.json
ok frame_index=0 timestamp_sec=0.0 path=storage

## Review service outputs

In [7]:
def latest_or_none(pattern_dir, pattern):
    files = sorted(pattern_dir.glob(pattern))
    return files[-1] if files else None

final_dir = PROJECT_ROOT / "storage/vision/outputs/final_analysis"
handoff_dir = PROJECT_ROOT / "storage/vision/outputs/supervisor_handoff"

final_analysis_path = latest_or_none(final_dir, "final_analysis_*.json")
supervisor_handoff_path = latest_or_none(handoff_dir, "vision_supervisor_handoff_*.json")

if final_analysis_path and supervisor_handoff_path is None:
    print("supervisor handoff output not found. Building from latest final_analysis...")
    run_command([sys.executable, "ai/vision/build_supervisor_handoff.py"], timeout=None)
    supervisor_handoff_path = latest_or_none(handoff_dir, "vision_supervisor_handoff_*.json")

for label, path in [
    ("final_analysis", final_analysis_path),
    ("supervisor_handoff", supervisor_handoff_path),
]:
    print(chr(10) + "##", label)
    if path is None:
        print("not found")
        continue

    print(path)
    data = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(data, ensure_ascii=False, indent=2)[:4000])



## final_analysis
/workspace/SKN27-FINAL-3Team/storage/vision/outputs/final_analysis/final_analysis_bb_3_190909_pedestrian_226_21450.json
{
  "schema_version": "vision-final-analysis-v1",
  "status": "success",
  "analysis_scope": "single_video_poc",
  "vision_agent_output": {
    "agent_output": {
      "node_code": "accident_situation_analysis",
      "status": "success",
      "summary": "video key frame에서 car 15건, person 1건이 탐지되었습니다. 2.933~9.467초 구간이 우선 확인 후보입니다. 이 결과는 관찰 근거이며 과실비율, 가해 차량, 법적 책임을 확정하지 않습니다.",
      "structured_result": {
        "media_type": "video",
        "event_window_candidates": [
          {
            "event_candidate_id": "event_window_01",
            "event_window_start_sec": 2.933,
            "event_window_end_sec": 9.467,
            "priority_score": 1.0,
            "source_refs": [
              "frame_03",
              "frame_04"
            ],
            "basis": "bbox_motion_peak_with_2sec_context",
            "clip_status": "candidate_for